In [ ]:
#@title 1. Install System Dependencies
!pip -q install flask pyngrok torch torchvision pillow

In [ ]:
#@title 2. Upload Web Assets & Deep Learning Model
import os
import zipfile
from google.colab import files

print("Please select and upload app.py, cnn_model.pth, and your templates ZIP folder:")
uploaded = files.upload()

# Unzip UI layouts and static dependencies if provided
for filename in uploaded:
    if filename.lower().endswith(".zip"):
        with zipfile.ZipFile(filename, "r") as archive:
            archive.extractall("/content")
        print(f"Extracted: {filename} -> /content")

In [ ]:
#@title 3. Environment Integrity Verification
app_path = "/content/app.py"
model_path = "/content/cnn_model.pth"
html_path = "/content/templates/index.html"

if not os.path.exists(app_path):
    raise FileNotFoundError("Missing file: Please upload your Flask Python file and name it 'app.py'.")

if not os.path.exists(model_path):
    raise FileNotFoundError("Missing file: Please upload your trained network weights and name it 'cnn_model.pth'.")

if not os.path.exists(html_path):
    raise FileNotFoundError(
        "Missing templates/index.html. Ensure your uploaded ZIP contains a 'templates' directory with your base HTML layout."
    )

print("✅ Success: All deployment files are verified and correctly positioned.")

In [ ]:
#@title 4. Authenticate & Initialize Ngrok Tunnel
from pyngrok import ngrok
import getpass

# Secure token prompt using masked input instead of open plain text logs
NGROK_AUTH_TOKEN = getpass.getpass("Paste your ngrok authentication token: ").strip()

if not NGROK_AUTH_TOKEN:
    raise ValueError("An ngrok authentication token is mandatory to establish an external tunnel.")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

try:
    ngrok.kill()
except Exception:
    pass

print("✅ Ngrok network configurations successfully initialized.")

In [ ]:
#@title 5. Launch Flask Server & Expose App URL
import subprocess
import sys

# Run the application as an isolated background task
process = subprocess.Popen(
    ["python", app_path],
    cwd="/content",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

try:
    # Attach a secure forwarding proxy to local web server port 5000
    public_url = ngrok.connect(5000).public_url
    print("\n" + "="*60)
    print("🍅 TOMATO LEAF DISEASE DETECTION PORTAL ONLINE 🍅")
    print("="*60)
    print(f"Live Deployment Web Address: {public_url}")
    print("="*60)
    print("\n⚠️ Attention: Leave this cell running continuously. Terminating execution will break the public link.")
except Exception as err:
    process.terminate()
    raise RuntimeError(f"Tunnel assembly aborted. Connection breakdown details: {err}")